# Final Exam: Phase 3: CIFAR-100 with SGD (lr=0.10, lr=0.15)
Tập trung thực nghiệm trên **CIFAR-100** sử dụng thuật toán **SGD** với kiến trúc **TickNet-L v1** (200 epochs).

### Hướng dẫn chạy nhanh trên Kaggle:
1. Chọn **Accelerator**: GPU T4 x2 hoặc GPU P100.
2. Bật **Internet**: Always on (để clone repo và tải dataset).
3. Bấm **Save Version** -> **Save & Run All (Commit)** hoặc chạy từng cell.
4. Sau khi hoàn thành, file `phase3_cifar100_sgd_results.zip` sẽ tự động được tạo ở `/kaggle/working/` để tải về máy.


In [ ]:
# 1. Environment & GPU Preflight Check
import os
import torch

# Auto-clone repository from branch feature/final-exam-model-l
if not os.path.exists('train_cifar.py'):
    !test -d /kaggle/working/TickNets/.git || git clone --depth 1 --branch feature/final-exam-model-l https://github.com/khuonglaptri3/TickNets.git /kaggle/working/TickNets
    %cd /kaggle/working/TickNets

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
!nvidia-smi


In [ ]:
# 2. Download and Verify CIFAR-100 Dataset
!python download_cifar.py --data-root /kaggle/working/data --dataset cifar100


### Thực nghiệm 1: SGD lr=0.10 + Nesterov 0.9, Cutout 16x16 (Weight Decay 1e-4)

In [ ]:
# Run cifar100_sgd_lr010
!python train_cifar.py --config configs/final/cifar100_sgd_lr010.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar100_sgd_lr010


### Thực nghiệm 2: SGD lr=0.15 + Nesterov 0.9, Cutout 16x16 (Weight Decay 1e-4)

In [ ]:
# Run cifar100_sgd_lr015
!python train_cifar.py --config configs/final/cifar100_sgd_lr015.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar100_sgd_lr015


### Tổng kết Kết Quả và Đóng Gói Artifacts

In [ ]:
import json
import shutil
from pathlib import Path
import pandas as pd

runs_dir = Path('/kaggle/working/runs')
target_configs = ['cifar100_sgd_lr010', 'cifar100_sgd_lr015']
records = []

for exp_name in target_configs:
    p = runs_dir / exp_name
    metric_file = p / 'test_metrics.json'
    cfg_file = p / 'config.json'
    if metric_file.is_file() and cfg_file.is_file():
        metrics = json.loads(metric_file.read_text())
        cfg = json.loads(cfg_file.read_text())
        records.append({
            'Experiment': exp_name,
            'Dataset': cfg['dataset'].upper(),
            'Optimizer': cfg['optimizer'].upper(),
            'LR': cfg['learning_rate'],
            'Test Top-1 (%)': f"{metrics['top1']:.2f}%",
            'Test Loss': f"{metrics['loss']:.4f}",
            'Macro F1': f"{metrics['macro_f1']:.4f}",
            'Params': f"{cfg['learnable_parameters']:,}",
            'FLOPs (G)': f"{cfg['gflops_forward']:.4f}G",
        })

df = pd.DataFrame(records)
print('=== KẾT QUẢ THỰC NGHIỆM ===')
display(df)

# Đóng gói kết quả thành file zip để download
zip_base = Path('/kaggle/working') / 'phase3_cifar100_sgd_results'
stage_dir = Path('/kaggle/working/stage_results')
if stage_dir.exists():
    shutil.rmtree(stage_dir)
stage_dir.mkdir(parents=True, exist_ok=True)
for exp in target_configs:
    src = runs_dir / exp
    if src.exists():
        shutil.copytree(src, stage_dir / exp)

shutil.make_archive(str(zip_base), 'zip', stage_dir)
print(f'[✓] Đã tạo file kết quả tải về tại: {zip_base}.zip')
